# Understanding Cross-Validation Metrics

This notebook provides a deep dive into the metrics used for evaluating time series forecasting models.

## What you'll learn:
- What sCRPS measures and why it's our primary metric
- How to interpret MAE, RMSE, and their relationship
- Understanding prediction interval coverage
- Detecting model miscalibration
- Comparing distributional vs quantile models

## 1. Setup and Create Sample Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import sys
sys.path.append('../..')

# Project imports
from uq.metrics import compute_scrps, compute_coverage_by_level
from uq.calibration import compute_pit

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

In [ ]:
# Generate synthetic forecast data for demonstration
np.random.seed(42)
n_samples = 1000

# True values (synthetic returns)
y_true = np.random.randn(n_samples) * 0.01

# Create different quality predictions
# Good model - small error, well-calibrated
good_pred = y_true + np.random.randn(n_samples) * 0.002
good_std = np.abs(np.random.randn(n_samples) * 0.003 + 0.003)

# Overconfident model - underestimates uncertainty
overconf_pred = y_true + np.random.randn(n_samples) * 0.002
overconf_std = np.abs(np.random.randn(n_samples) * 0.001 + 0.001)  # Too narrow

# Underconfident model - overestimates uncertainty  
underconf_pred = y_true + np.random.randn(n_samples) * 0.002
underconf_std = np.abs(np.random.randn(n_samples) * 0.008 + 0.008)  # Too wide

# Biased model - systematic error
biased_pred = y_true + 0.003 + np.random.randn(n_samples) * 0.002  # Positive bias
biased_std = np.abs(np.random.randn(n_samples) * 0.003 + 0.003)

print(f"Created {n_samples} synthetic predictions for 4 model types")

## 2. Understanding sCRPS (Scaled Continuous Ranked Probability Score)

sCRPS is our primary metric because it evaluates the entire predictive distribution, not just point forecasts.

In [ ]:
def compute_crps_gaussian(y_true, mean, std):
    """Compute CRPS for Gaussian distribution."""
    # Standardize
    z = (y_true - mean) / std
    
    # CRPS formula for Gaussian
    pdf_z = stats.norm.pdf(z)
    cdf_z = stats.norm.cdf(z)
    
    crps = std * (z * (2 * cdf_z - 1) + 2 * pdf_z - 1/np.sqrt(np.pi))
    return crps

# Compute CRPS for each model
models = {
    'Good Model': (good_pred, good_std),
    'Overconfident': (overconf_pred, overconf_std),
    'Underconfident': (underconf_pred, underconf_std),
    'Biased Model': (biased_pred, biased_std)
}

print("=" * 60)
print("CRPS and sCRPS Comparison")
print("=" * 60)

# Compute MAE for scaling
mae_baseline = np.mean(np.abs(y_true - np.mean(y_true)))

results = {}
for name, (pred, std) in models.items():
    crps = compute_crps_gaussian(y_true, pred, std)
    mean_crps = np.mean(crps)
    
    # Scale by MAE to get sCRPS
    scrps = mean_crps / mae_baseline
    
    results[name] = {
        'CRPS': mean_crps,
        'sCRPS': scrps,
        'MAE': np.mean(np.abs(y_true - pred))
    }
    
    print(f"\n{name:15s}: CRPS={mean_crps:.6f}, sCRPS={scrps:.3f}, MAE={results[name]['MAE']:.6f}")

In [ ]:
# Visualize sCRPS components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: sCRPS comparison
ax = axes[0]
model_names = list(results.keys())
scrps_values = [results[m]['sCRPS'] for m in model_names]
colors = ['green', 'red', 'orange', 'purple']

bars = ax.bar(range(len(model_names)), scrps_values, color=colors, alpha=0.7)
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=45)
ax.set_ylabel('sCRPS')
ax.set_title('sCRPS by Model Type\n(Lower is Better)')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Good threshold')
ax.legend()

# Plot 2: CRPS vs MAE relationship
ax = axes[1]
for name, color in zip(model_names, colors):
    ax.scatter(results[name]['MAE'], results[name]['CRPS'], 
               s=100, color=color, label=name, alpha=0.7)

ax.set_xlabel('MAE')
ax.set_ylabel('CRPS')
ax.set_title('CRPS vs MAE\n(CRPS accounts for uncertainty)')
ax.legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: CRPS penalizes both poor accuracy (MAE) and poor uncertainty estimates.")
print("The overconfident model has low MAE but higher CRPS due to narrow intervals.")
print("The underconfident model is penalized for intervals that are too wide.")

## 3. Understanding MAE and RMSE Relationship

In [ ]:
# Compute MAE and RMSE for each model
print("=" * 60)
print("MAE vs RMSE Analysis")
print("=" * 60)

for name, (pred, _) in models.items():
    errors = y_true - pred
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    ratio = rmse / mae
    
    print(f"\n{name:15s}:")
    print(f"  MAE:       {mae:.6f}")
    print(f"  RMSE:      {rmse:.6f}")
    print(f"  RMSE/MAE:  {ratio:.3f}")
    
    # Interpret the ratio
    if ratio < 1.15:
        print(f"  → Errors are relatively uniform")
    elif ratio < 1.35:
        print(f"  → Errors are approximately normal")
    else:
        print(f"  → Heavy-tailed errors or outliers present")

In [ ]:
# Visualize error distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, (pred, _)) in enumerate(models.items()):
    ax = axes[idx]
    errors = y_true - pred
    
    # Histogram with KDE
    ax.hist(errors, bins=30, alpha=0.6, density=True, label='Errors')
    
    # Overlay normal distribution
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, stats.norm.pdf(x, errors.mean(), errors.std()), 
            'r-', label='Normal fit')
    
    # Add vertical lines for MAE and RMSE
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    ax.axvline(-mae, color='blue', linestyle='--', alpha=0.5, label=f'±MAE')
    ax.axvline(mae, color='blue', linestyle='--', alpha=0.5)
    ax.axvline(-rmse, color='green', linestyle='--', alpha=0.5, label=f'±RMSE')
    ax.axvline(rmse, color='green', linestyle='--', alpha=0.5)
    
    ax.set_title(f'{name}\nRMSE/MAE = {rmse/mae:.3f}')
    ax.set_xlabel('Prediction Error')
    ax.set_ylabel('Density')
    ax.legend(loc='upper right')

plt.suptitle('Error Distribution Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Coverage Analysis

Coverage measures how often the true values fall within prediction intervals.

In [ ]:
# Compute prediction intervals and coverage
def compute_intervals_and_coverage(pred, std, y_true, levels=[80, 90, 95]):
    """Compute prediction intervals and their coverage."""
    coverage = {}
    intervals = {}
    
    for level in levels:
        # Compute z-score for the level
        alpha = (100 - level) / 100
        z = stats.norm.ppf(1 - alpha/2)
        
        # Compute intervals
        lower = pred - z * std
        upper = pred + z * std
        
        # Check coverage
        covered = (y_true >= lower) & (y_true <= upper)
        coverage[level] = np.mean(covered)
        intervals[level] = (lower, upper)
    
    return coverage, intervals

print("=" * 60)
print("COVERAGE ANALYSIS")
print("=" * 60)

for name, (pred, std) in models.items():
    coverage, _ = compute_intervals_and_coverage(pred, std, y_true)
    
    print(f"\n{name}:")
    for level, cov in coverage.items():
        actual_pct = cov * 100
        target_min = level - 2
        target_max = level + 2
        
        # Check calibration
        if target_min <= actual_pct <= target_max:
            status = "✓ Well calibrated"
        elif actual_pct < target_min:
            status = "✗ Undercover (overconfident)"
        else:
            status = "✗ Overcover (underconfident)"
        
        print(f"  {level}% PI: {actual_pct:.1f}% coverage {status}")

In [ ]:
# Visualize coverage reliability
fig, ax = plt.subplots(figsize=(10, 8))

# Nominal levels to test
nominal_levels = np.arange(10, 100, 5)

for name, (pred, std) in models.items():
    empirical_coverage = []
    
    for level in nominal_levels:
        cov, _ = compute_intervals_and_coverage(pred, std, y_true, [level])
        empirical_coverage.append(cov[level] * 100)
    
    ax.plot(nominal_levels, empirical_coverage, 'o-', label=name, alpha=0.7)

# Add diagonal line for perfect calibration
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Perfect calibration')

# Add tolerance bands
ax.fill_between([0, 100], [-2, 98], [2, 102], 
                alpha=0.1, color='gray', label='±2% tolerance')

ax.set_xlabel('Nominal Coverage (%)')
ax.set_ylabel('Empirical Coverage (%)')
ax.set_title('Coverage Reliability Diagram')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

plt.show()

print("\nInterpretation:")
print("- Points above diagonal: Model is underconfident (intervals too wide)")
print("- Points below diagonal: Model is overconfident (intervals too narrow)")
print("- Points on diagonal: Perfect calibration")

## 5. Bias Detection

In [ ]:
# Analyze bias in predictions
print("=" * 60)
print("BIAS ANALYSIS")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (pred, _) in models.items():
    errors = pred - y_true
    
    # Compute bias metrics
    mean_bias = np.mean(errors)
    median_bias = np.median(errors)
    
    # Test for significant bias
    t_stat, p_value = stats.ttest_1samp(errors, 0)
    
    print(f"\n{name}:")
    print(f"  Mean bias:    {mean_bias:.6f}")
    print(f"  Median bias:  {median_bias:.6f}")
    print(f"  t-statistic:  {t_stat:.3f}")
    print(f"  p-value:      {p_value:.4f}")
    
    if p_value < 0.05:
        if mean_bias > 0:
            print(f"  → Significant positive bias (overestimating)")
        else:
            print(f"  → Significant negative bias (underestimating)")
    else:
        print(f"  → No significant bias detected")

# Visualize bias patterns
ax = axes[0]
for name, (pred, _) in models.items():
    errors = pred - y_true
    ax.scatter(y_true[:100], errors[:100], alpha=0.5, s=20, label=name)

ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('True Value')
ax.set_ylabel('Prediction Error')
ax.set_title('Error vs True Value (first 100 points)')
ax.legend()
ax.grid(True, alpha=0.3)

# Cumulative bias over time
ax = axes[1]
for name, (pred, _) in models.items():
    errors = pred - y_true
    cumulative_bias = np.cumsum(errors) / np.arange(1, len(errors) + 1)
    ax.plot(cumulative_bias, label=name, alpha=0.7)

ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Sample Number')
ax.set_ylabel('Cumulative Mean Bias')
ax.set_title('Cumulative Bias Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Comparing Distributional vs Quantile Predictions

In [ ]:
# Simulate distributional vs quantile predictions
np.random.seed(42)

# Distributional model (StudentT)
dist_loc = y_true + np.random.randn(n_samples) * 0.002
dist_scale = np.abs(np.random.randn(n_samples) * 0.003 + 0.003)
dist_df = 4  # Degrees of freedom for Student-t

# Quantile model predictions
quantiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
quantile_preds = {}
for q in quantiles:
    # Generate quantile predictions with some noise
    z = stats.norm.ppf(q)
    quantile_preds[q] = dist_loc + z * dist_scale * (1 + np.random.randn(n_samples) * 0.1)

print("Created distributional (StudentT) and quantile model predictions")

In [ ]:
# Compare sCRPS computation methods
print("=" * 60)
print("DISTRIBUTIONAL vs QUANTILE MODELS")
print("=" * 60)

# Compute sCRPS for distributional model
dist_crps = []
for i in range(n_samples):
    # For Student-t, we'd use the actual CDF
    # Here we approximate with normal for simplicity
    crps = compute_crps_gaussian(y_true[i], dist_loc[i], dist_scale[i])
    dist_crps.append(crps)

dist_scrps = np.mean(dist_crps) / mae_baseline

print(f"\nDistributional Model (StudentT):")
print(f"  sCRPS: {dist_scrps:.4f}")
print(f"  Advantages:")
print(f"    - Full probability distribution")
print(f"    - Can compute any quantile")
print(f"    - Natural uncertainty quantification")
print(f"    - Supports PIT analysis")

# Compute sCRPS for quantile model (using trapezoidal rule)
def compute_quantile_crps(y_true, quantile_preds, quantiles):
    """Compute CRPS from quantile predictions using pinball loss."""
    crps = 0
    for q, pred in zip(quantiles, quantile_preds):
        errors = y_true - pred
        pinball = np.where(errors >= 0, q * errors, (q - 1) * errors)
        crps += np.mean(pinball)
    return crps / len(quantiles)

# Convert dict to array for computation
q_preds_array = np.column_stack([quantile_preds[q] for q in quantiles])
quantile_crps = []
for i in range(n_samples):
    crps = compute_quantile_crps(y_true[i], q_preds_array[i], quantiles)
    quantile_crps.append(crps)

quantile_scrps = np.mean(quantile_crps) / mae_baseline

print(f"\nQuantile Model (MQLoss):")
print(f"  sCRPS: {quantile_scrps:.4f}")
print(f"  Advantages:")
print(f"    - No distribution assumptions")
print(f"    - Direct quantile optimization")
print(f"    - Flexible for asymmetric risks")
print(f"    - Robust to outliers")

In [ ]:
# Visualize distributional vs quantile predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Select a subset for visualization
idx_start, idx_end = 100, 150
x_range = range(idx_start, idx_end)

# Plot distributional model
ax = axes[0]
ax.plot(x_range, y_true[idx_start:idx_end], 'k-', label='True', alpha=0.7)
ax.plot(x_range, dist_loc[idx_start:idx_end], 'b-', label='Mean', alpha=0.7)

# Add confidence bands for distributional model
for level, alpha in [(90, 0.3), (80, 0.4)]:
    z = stats.norm.ppf(1 - (100-level)/200)
    lower = dist_loc[idx_start:idx_end] - z * dist_scale[idx_start:idx_end]
    upper = dist_loc[idx_start:idx_end] + z * dist_scale[idx_start:idx_end]
    ax.fill_between(x_range, lower, upper, alpha=alpha, label=f'{level}% PI')

ax.set_title('Distributional Model (StudentT)')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot quantile model
ax = axes[1]
ax.plot(x_range, y_true[idx_start:idx_end], 'k-', label='True', alpha=0.7)
ax.plot(x_range, quantile_preds[0.5][idx_start:idx_end], 'b-', label='Median', alpha=0.7)

# Add quantile bands
ax.fill_between(x_range, 
                quantile_preds[0.1][idx_start:idx_end],
                quantile_preds[0.9][idx_start:idx_end],
                alpha=0.3, label='80% PI (q10-q90)')
ax.fill_between(x_range,
                quantile_preds[0.05][idx_start:idx_end],
                quantile_preds[0.95][idx_start:idx_end],
                alpha=0.2, label='90% PI (q05-q95)')

ax.set_title('Quantile Model (MQLoss)')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Metric Interpretation Guidelines

In [ ]:
# Create interpretation guide
print("=" * 70)
print("METRIC INTERPRETATION GUIDE")
print("=" * 70)

interpretation_guide = {
    'sCRPS': {
        'Excellent': '< 0.3',
        'Good': '0.3 - 0.5',
        'Acceptable': '0.5 - 0.7',
        'Poor': '> 0.7',
        'Note': 'Primary metric - balances accuracy and uncertainty'
    },
    'Coverage_80': {
        'Well Calibrated': '78% - 82%',
        'Slightly Off': '75% - 78% or 82% - 85%',
        'Miscalibrated': '< 75% or > 85%',
        'Note': 'Should capture 80% of observations'
    },
    'Coverage_90': {
        'Well Calibrated': '88% - 92%',
        'Slightly Off': '85% - 88% or 92% - 95%',
        'Miscalibrated': '< 85% or > 95%',
        'Note': 'Should capture 90% of observations'
    },
    'RMSE/MAE': {
        'Uniform Errors': '< 1.15',
        'Normal Errors': '1.15 - 1.35',
        'Heavy Tails': '> 1.35',
        'Note': 'Indicates error distribution shape'
    },
    'Bias': {
        'Unbiased': '|bias| < 0.0001',
        'Small Bias': '0.0001 - 0.0005',
        'Moderate Bias': '0.0005 - 0.001',
        'Large Bias': '> 0.001',
        'Note': 'Systematic over/under prediction'
    }
}

for metric, ranges in interpretation_guide.items():
    print(f"\n{metric}:")
    for category, value in ranges.items():
        if category != 'Note':
            print(f"  {category:20s}: {value}")
    print(f"  → {ranges['Note']}")

## 8. Model Selection Based on Metrics

In [ ]:
# Simulate metrics for multiple models
np.random.seed(42)

model_metrics = {
    'NHITS_T': {'sCRPS': 0.42, 'MAE': 0.0023, 'RMSE': 0.0031, 'Cov80': 0.81, 'Cov90': 0.91},
    'TiDE_T': {'sCRPS': 0.44, 'MAE': 0.0024, 'RMSE': 0.0032, 'Cov80': 0.79, 'Cov90': 0.89},
    'NBEATSx_T': {'sCRPS': 0.45, 'MAE': 0.0025, 'RMSE': 0.0033, 'Cov80': 0.82, 'Cov90': 0.92},
    'PatchTST_T': {'sCRPS': 0.43, 'MAE': 0.0023, 'RMSE': 0.0030, 'Cov80': 0.78, 'Cov90': 0.88},
    'NHITS_MQ': {'sCRPS': 0.46, 'MAE': 0.0024, 'RMSE': 0.0032, 'Cov80': 0.80, 'Cov90': 0.90},
    'TiDE_IQ': {'sCRPS': 0.47, 'MAE': 0.0025, 'RMSE': 0.0033, 'Cov80': 0.81, 'Cov90': 0.91},
}

# Convert to DataFrame for easier analysis
metrics_df = pd.DataFrame(model_metrics).T

print("Model Metrics Summary:")
display(metrics_df.round(4))

In [ ]:
# Multi-criteria model selection
def score_models(metrics_df, weights=None):
    """Score models based on multiple criteria."""
    if weights is None:
        weights = {
            'sCRPS': 0.5,      # Primary metric
            'Coverage': 0.3,   # Calibration
            'MAE': 0.2         # Point accuracy
        }
    
    scores = {}
    
    for model in metrics_df.index:
        # sCRPS score (lower is better, so invert)
        scrps_score = 1 / (1 + metrics_df.loc[model, 'sCRPS'])
        
        # Coverage score (penalize deviation from nominal)
        cov80_dev = abs(metrics_df.loc[model, 'Cov80'] - 0.80)
        cov90_dev = abs(metrics_df.loc[model, 'Cov90'] - 0.90)
        coverage_score = 1 - (cov80_dev + cov90_dev) / 2
        
        # MAE score (lower is better, so invert)
        mae_score = 1 / (1 + metrics_df.loc[model, 'MAE'] * 1000)
        
        # Weighted total
        total_score = (
            weights['sCRPS'] * scrps_score +
            weights['Coverage'] * coverage_score +
            weights['MAE'] * mae_score
        )
        
        scores[model] = {
            'sCRPS_score': scrps_score,
            'Coverage_score': coverage_score,
            'MAE_score': mae_score,
            'Total_score': total_score
        }
    
    return pd.DataFrame(scores).T.sort_values('Total_score', ascending=False)

# Score and rank models
model_scores = score_models(metrics_df)

print("=" * 70)
print("MODEL SELECTION ANALYSIS")
print("=" * 70)
print("\nScoring weights:")
print("  sCRPS: 50%")
print("  Coverage: 30%")
print("  MAE: 20%")

print("\nModel Rankings:")
display(model_scores.round(3))

print("\nRecommendations:")
best_overall = model_scores.index[0]
best_dist = [m for m in model_scores.index if '_T' in m][0]
best_quantile = [m for m in model_scores.index if '_MQ' in m or '_IQ' in m][0]

print(f"  Best Overall: {best_overall} (score: {model_scores.loc[best_overall, 'Total_score']:.3f})")
print(f"  Best Distributional: {best_dist}")
print(f"  Best Quantile: {best_quantile}")

## Summary

### Key Metrics and Their Interpretation:

1. **sCRPS (Primary Metric)**
   - Evaluates entire predictive distribution
   - Scale-free (normalized by MAE)
   - Lower is better (< 0.5 is good for financial data)
   - Balances accuracy and uncertainty quantification

2. **Coverage Metrics**
   - Measure calibration quality
   - Should be within ±2% of nominal levels
   - Undercoverage → overconfident model
   - Overcoverage → underconfident model

3. **MAE and RMSE**
   - MAE: Robust measure of typical error
   - RMSE: Sensitive to outliers
   - RMSE/MAE ratio indicates error distribution shape

4. **Bias**
   - Systematic over/under prediction
   - Should be close to zero
   - Test with t-test for significance

### Model Selection Guidelines:

1. **Primary Criterion**: Lowest sCRPS
2. **Secondary**: Good calibration (coverage within tolerance)
3. **Consider use case**:
   - Risk-sensitive → Prefer well-calibrated models
   - Point forecasting → Consider MAE/RMSE
   - Uncertainty critical → Distributional models

### Next Steps:

- See `03_calibration_diag.ipynb` for PIT analysis
- See `04_model_selection.ipynb` for ensemble strategies
- See `05_performance_tuning.ipynb` for optimization